# 03. Partición de los datos y modelos base

**Aplicación en el proyecto:** antes de entrenar cualquier GNN se hacen dos cosas.

1. **Separar los productos** en entrenamiento (70%), validación (15%) y prueba (15%). Los productos con el mismo árbol de ingredientes (por ejemplo, el mismo producto en dos tiendas) se mantienen juntos, para que el modelo no vea en el entrenamiento una copia de lo que luego se evalúa.
2. **Entrenar modelos base** que no usan grafos (promedio, tamaño del árbol, TF-IDF con Ridge y con Random Forest). Sus resultados son los números que las GNN tendrán que superar.

El umbral de AUP (`UMBRAL_AUP = 0.5`) es provisional hasta confirmar cómo se define `f_FPro_class`.

In [8]:
import numpy as np
import pandas as pd
from pathlib import Path

SEMILLA = 42        # para que los resultados se puedan repetir
UMBRAL_AUP = 0.5    # PROVISIONAL: se confirmará con la definición de f_FPro_class

carpeta = Path("../data/procesado")
if (carpeta / "productos.parquet").exists():
    df_productos = pd.read_parquet(carpeta / "productos.parquet")
    df_nodos = pd.read_parquet(carpeta / "nodos.parquet")
else:
    df_productos = pd.read_csv(carpeta / "productos.csv", dtype={"original_ID": str})
    df_nodos = pd.read_csv(carpeta / "nodos.csv", dtype={"original_ID": str},
                           keep_default_na=False, na_values=[""])

print("Productos:", len(df_productos), "| Nodos:", len(df_nodos))
print("Proporción de AUP con umbral", UMBRAL_AUP, ":",
      round((df_productos["f_FPro"] >= UMBRAL_AUP).mean(), 3))

Productos: 22517 | Nodos: 406330
Proporción de AUP con umbral 0.5 : 0.844


## 1. Productos con el mismo árbol de ingredientes

El mismo producto puede aparecer en varias tiendas o tamaños con exactamente la misma lista de ingredientes. Si una copia queda en entrenamiento y otra en prueba, el modelo ya vio la respuesta. Por eso a cada producto se le asigna una "firma" (su árbol completo) y los productos con la misma firma se tratan como un grupo.

In [9]:
# Firma de cada producto = su árbol completo (nivel y nombre de cada ingrediente, en orden)
nodos_ord = df_nodos.sort_values(["original_ID", "id_nodo"]).copy()
nodos_ord["pieza"] = nodos_ord["profundidad"].astype(str) + ":" + nodos_ord["ingredient_name"].astype(str)
firma = nodos_ord.groupby("original_ID")["pieza"].agg("|".join)

df_productos["grupo"] = pd.factorize(df_productos["original_ID"].map(firma))[0]

n_grupos = df_productos["grupo"].nunique()
print("Productos:", len(df_productos), "| árboles distintos:", n_grupos)
print("Productos cuyo árbol se repite en otro producto:",
      int(df_productos.duplicated("grupo", keep=False).sum()))

Productos: 22517 | árboles distintos: 17592
Productos cuyo árbol se repite en otro producto: 7360


## 2. Partición en entrenamiento, validación y prueba

- **Entrenamiento (70%):** con estos productos aprenden los modelos.
- **Validación (15%):** sirve para elegir ajustes 
- **Prueba (15%):** se usa una sola vez al final para reportar el resultado.

Las comprobaciones `assert` verifican que ningún árbol aparezca en dos conjuntos distintos.

In [10]:
from sklearn.model_selection import GroupShuffleSplit

# 70% entrenamiento, 15% validación, 15% prueba, sin separar productos con el mismo árbol
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEMILLA)
i_train, i_resto = next(gss1.split(df_productos, groups=df_productos["grupo"]))

resto = df_productos.iloc[i_resto]
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEMILLA)
i_val, i_test = next(gss2.split(resto, groups=resto["grupo"]))

df_productos["split"] = ""
df_productos.iloc[i_train, df_productos.columns.get_loc("split")] = "train"
df_productos.iloc[i_resto[i_val], df_productos.columns.get_loc("split")] = "val"
df_productos.iloc[i_resto[i_test], df_productos.columns.get_loc("split")] = "test"

# Ningún árbol aparece en dos conjuntos distintos
grupos_por_split = df_productos.groupby("split")["grupo"].agg(set)
assert not (grupos_por_split["train"] & grupos_por_split["val"])
assert not (grupos_por_split["train"] & grupos_por_split["test"])
assert not (grupos_por_split["val"] & grupos_por_split["test"])
assert (df_productos["split"] != "").all()
print("Sin árboles repetidos entre conjuntos: OK\n")

df_productos["es_AUP"] = (df_productos["f_FPro"] >= UMBRAL_AUP).astype(int)
print(df_productos.groupby("split").agg(productos=("original_ID", "size"),
                                         FPro_promedio=("f_FPro", "mean"),
                                         proporcion_AUP=("es_AUP", "mean")).round(3))

df_productos[["original_ID", "split"]].to_csv(carpeta / "split.csv", index=False)
print("\nGuardado:", carpeta / "split.csv")

Sin árboles repetidos entre conjuntos: OK

       productos  FPro_promedio  proporcion_AUP
split                                          
test        3398          0.719           0.841
train      15605          0.719           0.849
val         3514          0.699           0.822

Guardado: ../data/procesado/split.csv


## 3. Función de evaluación

Calcula las mismas métricas para todos los modelos: **MAE** y **MSE** (error de la regresión) y, aplicando el umbral de AUP, **AUC**, **exactitud balanceada** y **recall y F1 de la clase minoritaria** (no-AUP), además del F1 de AUP. Con clases desbalanceadas, la exactitud global sola engaña (Sokolova y Lapalme, 2009).

In [11]:
from sklearn.metrics import (mean_absolute_error, mean_squared_error, roc_auc_score,
                             balanced_accuracy_score, precision_recall_fscore_support)

def evaluar(y_real, y_pred, nombre):
    """Calcula las métricas de regresión y de clasificación (con el umbral de AUP)."""
    y_real = np.asarray(y_real)
    y_pred = np.clip(y_pred, 0, 1)
    real = (y_real >= UMBRAL_AUP).astype(int)
    pred = (y_pred >= UMBRAL_AUP).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(real, pred, labels=[0, 1], zero_division=0)
    return {"modelo": nombre,
            "MAE": mean_absolute_error(y_real, y_pred),
            "MAE no-AUP": mean_absolute_error(y_real[real == 0], y_pred[real == 0]),
            "MAE AUP": mean_absolute_error(y_real[real == 1], y_pred[real == 1]),
            "MSE": mean_squared_error(y_real, y_pred),
            "AUC": roc_auc_score(real, y_pred),
            "Exactitud balanceada": balanced_accuracy_score(real, pred),
            "Recall no-AUP": rec[0], "F1 no-AUP": f1[0], "F1 AUP": f1[1]}

# Separamos los datos por conjunto
tr = df_productos[df_productos["split"] == "train"]
va = df_productos[df_productos["split"] == "val"]
te = df_productos[df_productos["split"] == "test"]
y_tr, y_va, y_te = tr["f_FPro"].values, va["f_FPro"].values, te["f_FPro"].values

resultados_val, resultados_test = [], []
def registrar(nombre, pred_val, pred_test):
    resultados_val.append(evaluar(y_va, pred_val, nombre))
    resultados_test.append(evaluar(y_te, pred_test, nombre))

## 4. Dos modelos muy sencillos

- **Promedio:** siempre predice el FPro promedio del entrenamiento. Cualquier modelo útil tiene que ganarle.
- **Tamaño del árbol:** solo conoce cuántos nodos tiene el árbol y qué tan profundo es. Ravandi et al. (2025) observaron que los AUP tienen árboles más complejos, así que si este modelo ya predice bien, la GNN tendrá que demostrar que aporta algo más que el tamaño.

In [12]:
from sklearn.linear_model import LinearRegression

# Modelo 1: siempre predice el FPro promedio del entrenamiento
registrar("Promedio",
          np.full(len(y_va), y_tr.mean()), np.full(len(y_te), y_tr.mean()))

# Modelo 2: solo conoce el tamaño del árbol (cuántos nodos y qué tan profundo)
cols = ["total_nodos", "profundidad_max"]
tam = LinearRegression().fit(tr[cols], y_tr)
registrar("Tamaño del árbol", tam.predict(va[cols]), tam.predict(te[cols]))

pd.DataFrame(resultados_val).round(3)

,modelo,MAE,MAE no-AUP,MAE AUP,MSE,AUC,Exactitud balanceada,Recall no-AUP,F1 no-AUP,F1 AUP
0,Promedio,0.199,0.479,0.139,0.066,0.500,0.5,0.0,0.0,0.903
1,Tamaño del árbol,0.177,0.389,0.131,0.050,0.919,0.5,0.0,0.0,0.903


## 5. Modelo base de texto plano: TF-IDF + Ridge

Es la comparación que pidió el maestro. Cada producto se convierte en un texto con sus ingredientes (sin la raíz y **sin la estructura**: no importa quién contiene a quién) y TF-IDF mide qué palabras pesan más. Con ese texto se entrena una regresión Ridge. El vocabulario se aprende solo con el conjunto de entrenamiento.

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge

# Cada producto se convierte en un texto: sus ingredientes, sin la raíz y sin estructura
nodos_texto = df_nodos[df_nodos["id_nodo"] > 0].sort_values(["original_ID", "id_nodo"])
texto = nodos_texto.groupby("original_ID")["ingredient_name"].agg(" ".join)
t_tr, t_va, t_te = (texto.reindex(d["original_ID"]).fillna("") for d in (tr, va, te))

# El vectorizador SOLO aprende el vocabulario del conjunto de entrenamiento
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_features=20000, sublinear_tf=True)
X_tr = tfidf.fit_transform(t_tr)
X_va, X_te = tfidf.transform(t_va), tfidf.transform(t_te)
print("Matriz de entrenamiento:", X_tr.shape)

# Elegimos alpha con el conjunto de validación
mejor = None
for alpha in [0.3, 1, 3, 10]:
    modelo = Ridge(alpha=alpha).fit(X_tr, y_tr)
    mae = mean_absolute_error(y_va, np.clip(modelo.predict(X_va), 0, 1))
    print(f"alpha={alpha}: MAE validación = {mae:.4f}")
    if mejor is None or mae < mejor[0]:
        mejor = (mae, alpha, modelo)

_, alpha_ridge, ridge = mejor
print("Mejor alpha:", alpha_ridge)
registrar("TF-IDF + Ridge", ridge.predict(X_va), ridge.predict(X_te))

Matriz de entrenamiento: (15605, 11613)
alpha=0.3: MAE validación = 0.0927
alpha=1: MAE validación = 0.0929
alpha=3: MAE validación = 0.0982
alpha=10: MAE validación = 0.1090
Mejor alpha: 0.3


## 6. Modelo base de texto plano: TF-IDF + Random Forest

El mismo texto, pero con un bosque aleatorio, que es el tipo de modelo que usa FoodProX. Puede tardar varios minutos; si se alarga, se baja `n_estimators`.

In [14]:
from sklearn.ensemble import RandomForestRegressor

# Versión más pequeña del vocabulario para que el bosque no tarde demasiado
tfidf_rf = TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_features=3000, sublinear_tf=True)
Xr_tr = tfidf_rf.fit_transform(t_tr)
Xr_va, Xr_te = tfidf_rf.transform(t_va), tfidf_rf.transform(t_te)

bosque = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=SEMILLA)
bosque.fit(Xr_tr, y_tr)
registrar("TF-IDF + Random Forest", bosque.predict(Xr_va), bosque.predict(Xr_te))
print("Listo")

Listo


## 7. Prueba de las etiquetas revueltas

Se entrena el mismo modelo con los FPro mezclados al azar entre productos. Como ahora no hay relación real entre ingredientes y puntaje, el error debe quedar casi igual al de predecir el promedio. Si fuera mucho mejor, habría una fuga de datos en el proceso.

In [15]:
# Entrenamos con las etiquetas revueltas al azar: el modelo no puede aprender nada real
rng = np.random.default_rng(SEMILLA)
revuelto = Ridge(alpha=alpha_ridge).fit(X_tr, rng.permutation(y_tr))

mae_revuelto = mean_absolute_error(y_va, np.clip(revuelto.predict(X_va), 0, 1))
mae_promedio = mean_absolute_error(y_va, np.full(len(y_va), y_tr.mean()))
print(f"MAE con etiquetas revueltas: {mae_revuelto:.4f}")
print(f"MAE prediciendo el promedio: {mae_promedio:.4f}")

MAE con etiquetas revueltas: 0.2127
MAE prediciendo el promedio: 0.1992


## 8. Resultados de los modelos base

La tabla de **prueba** es la que se reporta. Estos son los números que deberán superar las GCN y las GAT. La tabla de validación solo sirve para revisar.

In [16]:
tabla_val = pd.DataFrame(resultados_val).set_index("modelo").round(3)
tabla_test = pd.DataFrame(resultados_test).set_index("modelo").round(3)

print("=== VALIDACIÓN ===")
display(tabla_val)
print("\n=== PRUEBA (resultado final de cada modelo base) ===")
display(tabla_test)

tabla_test.to_csv(carpeta / "resultados_modelos_base.csv")
print("\nGuardado:", carpeta / "resultados_modelos_base.csv")

=== VALIDACIÓN ===


,MAE,MAE no-AUP,MAE AUP,MSE,AUC,Exactitud balanceada,Recall no-AUP,F1 no-AUP,F1 AUP
modelo,,,,,,,,,
Promedio,0.199,0.479,0.139,0.066,0.500,0.500,0.000,0.000,0.903
Tamaño del árbol,0.177,0.389,0.131,0.050,0.919,0.500,0.000,0.000,0.903
TF-IDF + Ridge,0.093,0.162,0.078,0.016,0.964,0.887,0.811,0.817,0.961
TF-IDF + Random Forest,0.076,0.122,0.066,0.012,0.976,0.903,0.833,0.850,0.968



=== PRUEBA (resultado final de cada modelo base) ===


,MAE,MAE no-AUP,MAE AUP,MSE,AUC,Exactitud balanceada,Recall no-AUP,F1 no-AUP,F1 AUP
modelo,,,,,,,,,
Promedio,0.184,0.435,0.136,0.055,0.500,0.500,0.000,0.000,0.914
Tamaño del árbol,0.163,0.347,0.128,0.042,0.915,0.500,0.000,0.000,0.914
TF-IDF + Ridge,0.089,0.145,0.078,0.015,0.954,0.851,0.731,0.775,0.960
TF-IDF + Random Forest,0.073,0.119,0.064,0.011,0.963,0.898,0.820,0.844,0.971



Guardado: ../data/procesado/resultados_modelos_base.csv
